# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata_dict = json.loads(dataset.metadata.to_json())
print(f"{metadata_dict.get('name')}: {metadata_dict.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. The Croissant schema allows us to explicitly reference record sets, fields, and columns by their `@id`.

In [ ]:
# List all record sets in the dataset by their @id
record_sets = dataset.metadata.get('recordSet', [])

print('--- Record Sets ---')
for i, record_set in enumerate(record_sets):
    record_set_id = record_set.get('@id')
    record_set_name = record_set.get('name', '')
    print(f"{i+1}. @id: {record_set_id} | name: {record_set_name}")

print('\n')
# For each record set, list all fields by their @id
for record_set in record_sets:
    record_set_id = record_set.get('@id')
    print(f"Fields in Record Set @id={record_set_id}: {record_set.get('name', record_set_id)}")
    fields = record_set.get('field', [])
    # fields can be dict (single) or list
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"    Field @id: {field.get('@id')} | name: {field.get('name')}")
        elif isinstance(field, str):
            # If only @id is present as a string
            print(f"    Field @id: {field}")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# To keep this notebook robust, select record set(s) explicitly by @id

record_sets = dataset.metadata.get('recordSet', [])
assert len(record_sets) > 0, "No record sets found in metadata."

# Select the main data record set for protein quantification (the main data table)
# Inspect the record set IDs from the previous cell if in doubt.
main_record_set_id = record_sets[0].get('@id')  # Use the first record set for demonstration
print(f"Selected record set @id for data extraction: {main_record_set_id}")

dataframes = {}

# If you have more record sets to process, you can add their @id's to the list below
record_set_ids = [main_record_set_id]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))
    else:
        print(f"No records found for record_set_id={record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes outlier removal, data transformation, or grouping for further analysis.

All columns are referenced by their field or column `@id`.

In [ ]:
# For demonstration, we pick a numeric field/column by its @id.

# Show list of columns for reference
main_df = dataframes[main_record_set_id]
print(f"Columns in main DataFrame: {list(main_df.columns)}")

# Example: Let's try to analyze 'cr:coverage_percent' (protein sequence coverage percent)
numeric_field_id = 'cr:coverage_percent'
if numeric_field_id not in main_df.columns:
    print(f"Column {numeric_field_id} not found; using the first numeric column available.")
    # Try to find a likely numeric column
    for col in main_df.columns:
        if main_df[col].dtype.kind in 'fi':
            numeric_field_id = col
            print(f"Using numeric column: {col}")
            break

# Filter records
threshold = 10
if numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head(3))

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head(3))

    # Grouping by a category, e.g. 'cr:description'
    group_field = 'cr:description'
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
        print(grouped_df.head(3))
else:
    print(f"No suitable numeric column {numeric_field_id} found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in main_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group if group_field is present
    if group_field in main_df.columns:
        top_groups = main_df[group_field].value_counts().index[:5]
        sub_df = main_df[main_df[group_field].isin(top_groups)]
        plt.figure(figsize=(10,5))
        sns.boxplot(data=sub_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration. In this notebook, you:
- Loaded dataset metadata and records using Croissant schema via `mlcroissant`.
- Inspected available record sets and their fields, referencing all data entities by their `@id`.
- Extracted records into DataFrames for processing.
- Performed exploratory filtering, normalization, and grouping on numeric fields.
- Visualized data distributions and groupings.

Continue your analysis using field and record set `@id`s for reproducibility across updates to the dataset.